<a href="https://colab.research.google.com/github/devvrushabh/Hadoop-Pyspark/blob/main/2_Pyspark_RDD_to_DataFrame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("PySparkDemo") \
    .master("local[*]") \
    .getOrCreate()

## saveAsTextFile(path) --> Saves the RDD to a text file at the specified path

In [4]:
path = '/content/emp.txt'
output_path = '/content/agg_employ_data/'

rdd = spark.sparkContext.textFile(path)

# skip header
header = rdd.first()

res = rdd.filter(lambda x:x!=header )

# select name, sum(salary) from employ group by name

newrdd = res.map(lambda x: x.split(',')).map(lambda x: (x[0],int(x[2]))).reduceByKey(lambda x,y: x+y)

# newrdd.saveAsTextFile(output_path)

for i in newrdd.collect():
  print(i)


('pooja', 146000)
('vinod', 94000)
('akash', 102000)
('akshay', 45000)
('ram', 34000)


In [5]:
input_path = '/content/agg_employ_data/'

rdd = spark.sparkContext.textFile(input_path)


for i in rdd.collect():
  print(i)

('vinod', 94000)
('akash', 102000)
('akshay', 45000)
('ram', 34000)
('pooja', 146000)


In [6]:
path = '/content/emp.txt'

rdd = spark.sparkContext.textFile(path)

# skip header
header = rdd.first()

res = rdd.filter(lambda x:x!=header )

newrdd = res.map(lambda x: x.split(','))

for i in newrdd.collect():
  print(i)

['vinod', '1998', '67000', '500', 'Bombay']
['pooja', '2000', '68000', '670', 'Hyderabad']
['akash', '1998', '56000', '500', 'Bombay']
['pooja', '2002', '78000', '50', 'Hyderabad']
['akshay', '2020', '45000', '100', 'Pune']
['vinod', '1999', '27000', '100', 'Bombay']
['ram', '2018', '34000', '150', 'Hyderabad']
['akash', '2004', '46000', '60', 'Bombay']


## map() --> is a narrow transformation function applies on each row of dataset
## flatMap() --> is a narrow transformation operation that flattens the RDD after applying the function on every element

In [8]:
path = '/content/us500.csv'

rdd = spark.sparkContext.textFile(path)

# Fetch only email ids
nrdd = rdd.flatMap(lambda x: x.split('|')).filter(lambda x: '@' in x)

for i in nrdd.collect():
  print(i)

jbutt@gmail.com
josephine_darakjy@darakjy.org
art@venere.org
lpaprocki@hotmail.com
donette.foller@cox.net
simona@morasca.com
mitsue_tollner@yahoo.com
leota@hotmail.com
sage_wieser@cox.net
kris@gmail.com
minna_amigon@yahoo.com
amaclead@gmail.com
kiley.caldarera@aol.com
gruta@cox.net
calbares@gmail.com
mattie@aol.com
meaghan@hotmail.com
gladys.rim@rim.org
yuki_whobrey@aol.com
fletcher.flosi@yahoo.com
bette_nicka@cox.net
vinouye@aol.com
willard@hotmail.com
mroyster@royster.com
alisha@slusarski.com
allene_iturbide@cox.net
chanel.caudy@caudy.org
ezekiel@chui.com
wkusko@yahoo.com
bfigeroa@aol.com
ammie@corrio.com
francine_vocelka@vocelka.com
ernie_stenseth@aol.com
albina@glick.com
asergi@gmail.com
solange@shinko.com
jose@yahoo.com
rozella.ostrosky@ostrosky.com
valentine_gillian@gmail.com
kati.rulapaugh@hotmail.com
youlanda@aol.com
doldroyd@aol.com
roxane@hotmail.com
lperin@perin.org
erick.ferencz@aol.com
fsaylors@saylors.org
jina_briddick@briddick.com
kanisha_waycott@yahoo.com
emerson.bowley

In [ ]:
James|Butt|BentonJohn B Jr|6649 N Blue Gum St|New Orleans|Orleans|LA|70116|504-621-8927|504-845-1427|jbutt@gmail.com|http://www.bentonjohnbjr.com

James
Butt
BentonJohn B Jr
6649 N Blue Gum St
New Orleans
Orleans
LA
70116
504-621-8927
504-845-1427
jbutt@gmail.com


# RDD to --> Dataframe
# By default rdd is unstructured , so make it structured & convert it inro dataframe

## 2 ways to convert rdd into dataframe . but data must be structured

## 1) toDF() function --> rdd.toDF(columns_list)

## 2) createDataFrame function --> spark.createDataFrame(rdd, schema= colmns_list)


In [9]:
import re

path = '/content/bank-full.csv'

rdd = spark.sparkContext.textFile(path)

header = rdd.first()

res = rdd.filter(lambda x: x!=header).map(lambda x: re.sub('"','',x) ).map(lambda x: x.split(';'))

cols = ['age','job','marital','education','default','balance','housing','loan','contact','day','month','duration','campaign','pdays','previous','poutcome','y']

# df = res.toDF(cols)
df = spark.createDataFrame(res, schema = cols)

newdf = df.filter((df.marital=='single') & (df.balance > 1000) )

newdf.show()

# for i in res.take(100):
#   print(i)

+---+------------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|age|         job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+------------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
| 35| blue-collar| single|secondary|     no|  12223|    yes| yes|unknown|  5|  may|     177|       1|   -1|       0| unknown| no|
| 38|  technician| single|secondary|     no|   1685|    yes|  no|unknown|  5|  may|     185|       1|   -1|       0| unknown| no|
| 30|  technician| single|secondary|     no|   2573|    yes|  no|unknown|  5|  may|      67|       2|   -1|       0| unknown| no|
| 58|  management| single| tertiary|     no|   1387|    yes|  no|unknown|  5|  may|     174|       5|   -1|       0| unknown| no|
| 42|      admin.| single|secondary|     no|   1022|    yes|  no|unknown|  5|  may|     14

In [10]:
df.printSchema()

root
 |-- age: string (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: string (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: string (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- campaign: string (nullable = true)
 |-- pdays: string (nullable = true)
 |-- previous: string (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)



In [11]:
df.createOrReplaceTempView('employ')

In [12]:
spark.sql("select * from employ where marital='single' and balance>1000 ").show()

+---+------------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|age|         job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+------------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
| 35| blue-collar| single|secondary|     no|  12223|    yes| yes|unknown|  5|  may|     177|       1|   -1|       0| unknown| no|
| 38|  technician| single|secondary|     no|   1685|    yes|  no|unknown|  5|  may|     185|       1|   -1|       0| unknown| no|
| 30|  technician| single|secondary|     no|   2573|    yes|  no|unknown|  5|  may|      67|       2|   -1|       0| unknown| no|
| 58|  management| single| tertiary|     no|   1387|    yes|  no|unknown|  5|  may|     174|       5|   -1|       0| unknown| no|
| 42|      admin.| single|secondary|     no|   1022|    yes|  no|unknown|  5|  may|     14

In [13]:
spark.sql("select marital,sum(balance) as total_balance from employ group by marital").show()

+--------+-------------+
| marital|total_balance|
+--------+-------------+
|divorced|    6138388.0|
| married|  3.8805139E7|
|  single|  1.6646155E7|
+--------+-------------+



## RDD vs DataFrame
### 1) Rdd does not provide schema view of data. It has no provision for handling structured data. Mostly rdd used for handling unstructured data.

### 2) Dataframe is a data organized into rows and columns. It has schema view of data. It is conceptually equivalent to table in a relational database

# Use Below code in Databricks

### read CSV file in Dataframe

In [ ]:

path = '/Volumes/ai_dev/input/ai_dataset/us-500.csv'

df = spark.read.format('csv').options(header='true', inferSchema='true').load(path)
display(df)

## header --> It uses the 1st line as name of column
## inferSchema --> It instruct spark to automatically infer the data types of each column based on the content of data
# df.printSchema() # used to display the schema of a DataFrame in a tree format

In [ ]:
# you can also run sql queries on dataframe
df.createOrReplaceTempView('employ')

In [ ]:
newdf = spark.sql("SELECT * FROM employ where city='Brighton'")
display(newdf)

In [ ]:
%sql
select * from employ where county = 'Livingston'